In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import torch
import torch.nn as nn
import torchvision

In [10]:
m = nn.BatchNorm1d(2)
input = torch.randn(80, 2)
output = m(input)

output.shape

torch.Size([80, 2])

In [11]:
H, W = 32, 32

y_coords, x_coords = torch.meshgrid(
    torch.linspace(0, 1, H),
    torch.linspace(0, 1, W),
    indexing="ij"
)

In [12]:
def count_params(m):
    total_trainable_params = sum(
        p.numel() for p in m.parameters() if p.requires_grad
    )

    return total_trainable_params

In [13]:
# feature_extractor = torchvision.models.mobilenet_v3_small(
#     weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1,
# )


feature_extractor = torchvision.models.mobilenet_v3_large(
    weights=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2,
)

print(f'Initially {count_params(feature_extractor)}')

Initially 5483032


In [14]:
feature_extractor

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [6]:
print(f'Children Count {len(list(feature_extractor.children()))}')

Children Count 3


In [7]:
final_fe = list(feature_extractor.children())[0]
print(f'Children Count {len(list(final_fe.children()))}')
print(f'Trainable Params {count_params(final_fe)}')

Children Count 13
Trainable Params 927008


In [8]:
for param in final_fe.parameters():
    param.requires_grad = False 

for param in final_fe[-2:].parameters():
    param.requires_grad = True

print(f'For Fine Tuning {count_params(final_fe)}')

For Fine Tuning 350544


In [10]:
reference_patches = torch.randn(80, 3, 32, 32)

x = final_fe(reference_patches)
x.shape

torch.Size([80, 576, 1, 1])

In [12]:
dir(final_fe)

['T_destination',
 '__add__',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_item_by_idx',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_lo